# zsmedseg on Google Colab or Kaggle

Segment a medical image from a text description: **BMC-CLIP points, MedSAM2 draws.** Nothing is trained.

**Before you start:** choose a GPU.
* Colab: *Runtime > Change runtime type > T4 GPU*.
* Kaggle: in the right panel, *Accelerator > GPU T4 x2* (or P100), and switch **Internet on**.

Run the cells from top to bottom. Only cell 4 has something to choose: the dataset.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code
Set `REPO_URL` to your copy of the repository, or upload `zeroshot-medseg.zip` (Colab: the folder icon on the left;
Kaggle: attach it as a dataset) and leave `REPO_URL` empty.

In [ ]:
import glob, os, shutil

REPO_URL = ""  # e.g. "https://github.com/<you>/zeroshot-medseg.git"
ON_KAGGLE = os.path.isdir("/kaggle/working")
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
os.chdir(WORK)
if not os.path.isdir("zeroshot-medseg"):
    if REPO_URL:
        !git clone -q {REPO_URL} zeroshot-medseg
    else:
        zips = glob.glob(f"{WORK}/zeroshot-medseg.zip") + glob.glob("/kaggle/input/*/zeroshot-medseg.zip")
        assert zips, "Set REPO_URL or upload zeroshot-medseg.zip"
        !unzip -q -o {zips[0]} -d {WORK}
os.chdir(f"{WORK}/zeroshot-medseg")
print(os.getcwd())

## 2. Install
On Kaggle (Python 3.11) this installs the exact library versions behind the reported numbers. Colab uses
Python 3.12, where those versions do not exist; recent versions are used and the numbers differ slightly
(for example 83.93 instead of 83.11 Dice on the first 20 Kvasir images). If PyTorch is reinstalled,
restart the session once (*Runtime > Restart*) and run from the top. MedSAM2's code is cloned into the
repository folder, where it is found automatically.

In [ ]:
import sys
if sys.version_info < (3, 12):   # Kaggle: the exact versions behind the reported numbers
    !pip install -q torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu121
    !pip install -q -r requirements-exact.txt
else:                            # Colab (Python 3.12): recent versions, numbers differ slightly
    !pip install -q -r requirements.txt
!pip install -q --no-deps -e .
!pip install -q kaggle gdown "surface-distance @ git+https://github.com/google-deepmind/surface-distance.git"
![ -d MedSAM2 ] || git clone -q https://github.com/bowang-lab/MedSAM2.git

import sys
sys.path.insert(0, "src")  # lets this kernel import zsmedseg without a restart
!python -m zsmedseg --help

## 3. Model weights (about 5.3 GB, once)
BMC-CLIP (`BIOMEDICA/BMC_CLIP_CF`) and MedSAM2 (`wanglab/MedSAM2`) from the Hugging Face Hub. DINO downloads itself on first use.

In [ ]:
!python -m zsmedseg download-weights
!ls -lh weights

## 4. Choose and prepare a dataset

| `DATASET` | Target | Raw data | Source |
|---|---|---|---|
| `kvasir` | polyp | `kvasir` | public URL |
| `isic2018` | skin lesion | `isic2018` | public URL (2.4 GB) |
| `busi` | breast lesion | `busi` | Kaggle `aryashah2k/breast-ultrasound-images-dataset` |
| `refuge2` | optic disc | `refuge2` | Kaggle `victorlemosml/refuge2` |
| `camus` | left ventricle | `camus` | Kaggle `mykcs01/camus-public-202404` |
| `udiat`, `lung_xray`, `brain_mri`, `lung_ct` | tumour, lungs, tumour, lungs | `medclipsamv2` | Google Drive bundle (2.5 GB) |

**Kaggle datasets:** on Kaggle, attach the dataset with *Add Input* (the cell below finds it under `/kaggle/input`).
On Colab, upload your `kaggle.json` API token first (next cell).

In [ ]:
DATASET = "kvasir"   # <- choose from the table above

RAW = {"udiat": "medclipsamv2", "lung_xray": "medclipsamv2", "brain_mri": "medclipsamv2", "lung_ct": "medclipsamv2"}.get(DATASET, DATASET)
KAGGLE_SLUG = {"busi": "breast-ultrasound-images-dataset", "refuge2": "refuge2", "camus": "camus-public-202404"}.get(RAW)

attached = f"/kaggle/input/{KAGGLE_SLUG}" if KAGGLE_SLUG else None
if attached and os.path.isdir(attached):
    !python -m zsmedseg prepare-data {RAW} --source {attached}
else:
    if KAGGLE_SLUG and not ON_KAGGLE and not os.path.isfile(os.path.expanduser("~/.kaggle/kaggle.json")):
        from google.colab import files
        print("Upload kaggle.json (kaggle.com > Settings > API > Create New Token)")
        files.upload()
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        shutil.move("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    !python -m zsmedseg download-data {RAW}
    !python -m zsmedseg prepare-data {RAW}

## 5. Quick check: the first 20 images
Expected Dice: busi 81.43, kvasir 83.11, isic2018 87.16, refuge2 73.95, udiat 83.19, lung_xray 94.59,
brain_mri 47.19, lung_ct 2.53, camus 6.89. A different number means something differs (weights, data or library versions).

In [ ]:
!python -m zsmedseg evaluate -c configs/datasets/{DATASET}.yaml --set runtime.limit=20 --set runtime.save_overlays=true

In [ ]:
from IPython.display import Image, display
run = sorted(glob.glob("outputs/*/*/overlays"), key=os.path.getmtime)[-1]
print("image | target map + 5 points | prediction | ground truth")
for path in sorted(glob.glob(f"{run}/*.jpg"))[:3]:
    display(Image(filename=path, width=1000))

## 6. Full test set
Expected Dice: udiat 82.19, lung_xray 94.59, isic2018 90.40, kvasir 88.39, refuge2 81.57, busi 76.96
(and the known limitations brain_mri 49.54, lung_ct 12.88, camus 2.88). A T4 needs roughly 1 s per image.

In [ ]:
!python -m zsmedseg evaluate -c configs/datasets/{DATASET}.yaml

In [ ]:
import json
latest = sorted(glob.glob("outputs/*/*/metrics.json"), key=os.path.getmtime)[-1]
print(latest)
print(json.dumps(json.load(open(latest)), indent=2))

## 7. Your own images (no ground truth needed)
Describe the target and its surroundings in words. Results: `outputs/predictions/{masks,saliency,overlays}`.

In [ ]:
# Three images of the chosen dataset. For your own images, point --input at them and describe
# the target yourself, e.g.  --target "polyp" "colorectal polyp" --background "normal colon mucosa" "background"
from zsmedseg import load_config
images_dir = sorted(glob.glob(load_config([f"configs/datasets/{DATASET}.yaml"]).dataset.root + "/*images"))[0]
!python -m zsmedseg predict --input {images_dir} -c configs/datasets/{DATASET}.yaml --set runtime.limit=3
for path in sorted(glob.glob("outputs/predictions/overlays/*.jpg"))[:3]:
    display(Image(filename=path, width=900))

## 8. From Python
The models are loaded once; `set_prompts` switches the task in milliseconds.

In [ ]:
import cv2
from zsmedseg import ZeroShotSegmenter, load_config

cfg = load_config([f"configs/datasets/{DATASET}.yaml"])
segmenter = ZeroShotSegmenter.from_config(cfg)
image_path = sorted(glob.glob(f"{images_dir}/*"))[0]
result = segmenter.segment(cv2.imread(image_path))
print("points:", result.points)
print("15 candidates, predicted IoU:", [round(c.score, 3) for c in result.candidates])
print("kept:", [(k.prompt_index + 1, k.mask_index + 1) for k in result.kept], "| mask pixels:", int(result.mask.sum()))